In [85]:
from pathlib import Path
import json
import re


def load_benchmark_files(
    directory="benchmarks",
    throughput=None,
    n_scans_per_batch=None,
    n_workers=None,
    n_partitions=None,
    verify_content=True,
):
    """
    Load benchmark JSON files matching requested parameters.

    Parameters
    ----------
    directory : str
        Benchmark directory.
    throughput : int | None
        Required throughput in MBps. None means any.
    n_scans_per_batch : int | None
        Required scans per batch. None means any.
    n_partitions : int | None
        Required partition count. None means any.
    verify_content : bool
        If True, verify filename metadata matches JSON content.

    Returns
    -------
    list[dict]
        List of dicts with:
        {
            "path": Path,
            "throughput": int,
            "spb": int,
            "parts": int,
            "data": dict
        }
    """

    # Build glob pattern
    throughput_pattern = f"{throughput}MBps" if throughput is not None else "*MBps"
    
    nspb_pattern = f"{n_scans_per_batch}SpB" if n_scans_per_batch is not None else "*SpB"
    nparts_pattern = f"{n_partitions}parts" if n_partitions is not None else "*parts"
    nws_pattern = f"{n_workers}ws" if n_workers is not None else "*ws"

    pattern = f"*--{throughput_pattern}--{nspb_pattern}--{nws_pattern}--{nparts_pattern}.json"

    filename_re = re.compile(
        r".+--(?P<throughput>\d+)MBps--"
        r"(?P<spb>\d+)SpB--"
        r"(?P<ws>\d+)ws--"
        r"(?P<parts>\d+)parts\.json$"
    )

    results = []

    for path in Path(directory).glob(pattern):
        match = filename_re.match(path.name)
        if not match:
            continue

        filename_values = {
            "throughput_MB": int(match.group("throughput")),
            "n_scans_per_batch": int(match.group("spb")),
            "n_workers": int(match.group("ws")),
            "n_partitions": int(match.group("parts")),
        }

        with path.open() as f:
            content = json.load(f)

            # Adjust these keys if your JSON uses different names
            content_values = {
                "throughput_MB": content["throughput_MB"],
                "n_scans_per_batch": content["n_scans_per_batch"],
                "n_workers": content["n_workers"],
                "n_partitions": content["n_partitions"],
            }

            #if filename_values != content_values:
            #    raise ValueError(
            #        f"Filename/content mismatch:\n"
            #        f"File: {path}\n"
            #        f"Filename values: {filename_values}\n"
            #        f"JSON values: {content_values}"
            #    )

            results.append(
                {
                    "path": path,
                    **filename_values,
                    "data": content,
                })
    return results

In [86]:
benchmarks = load_benchmark_files(
    directory="benchmarks",
    throughput=16,
    n_partitions=3
)

for b in benchmarks:
    print(
        b["path"],
        b["n_scans_per_batch"],
        b["n_partitions"],
        b["n_workers"]
    )

# Benchmark metrics vs n_workers / partition_worker_ratio (generalized)

In [87]:
from pathlib import Path
from collections import defaultdict
import json as _json
import math

import matplotlib.pyplot as plt
import numpy as np


# From the Producer configuration (see producer.ipynb): each scan has
# N_SAMPLES_PER_SCAN=2048 samples, and each sample is an (I, Q) pair of
# float32 values -> 8 bytes/sample -> 16384 bytes/scan.
BYTES_PER_SCAN = 2048 * 8


def _dedup_records(records):
    """Defensive: keep only exact-duplicate protection in case some files
    still have it (older format runs had this - the current format doesn't,
    but this is a harmless no-op when there's nothing to remove)."""
    seen = set()
    unique = []
    for r in records:
        key = _json.dumps(r, sort_keys=True)
        if key not in seen:
            seen.add(key)
            unique.append(r)
    return unique


def _drop_stray_records(b):
    """Defensive: drop any record whose receive_ts falls outside this run's
    own [producer_begin_ts, analysis_end_ts] window (older format runs could
    carry over records from a previous run - the current format doesn't, but
    this is a harmless no-op when there's nothing to remove)."""
    begin = b["data"]["producer_begin_ts"]
    end = b["data"]["analysis_end_ts"]
    return [r for r in b["data"]["records"] if begin <= r["receive_ts"] <= end]


def _records(b):
    return _dedup_records(_drop_stray_records(b))


def _record_values(b, field):
    """One value per record for a scalar per-batch field (e.g.
    processing_time_ms, batch_latency_ms) - current format logs one batch
    per record, so no flattening of nested lists is needed anymore."""
    return [record[field] for record in _records(b)]


def _achieved_throughput_MBps(b):
    """Effective end-to-end throughput: data volume actually delivered to the
    consumer during THIS run, divided by the elapsed time. The current
    format reports n_total_scans directly, so no reconstruction from
    per-record batch counts is needed."""
    total_MB = b["data"]["n_total_scans"] * BYTES_PER_SCAN / (1024 ** 2)
    elapsed = b["data"]["analysis_end_ts"] - b["data"]["producer_begin_ts"]
    return total_MB / elapsed


# Registry of measurable quantities. Each metric only needs to describe how to
# extract a single scalar per benchmark run ("extract"), a y-axis label, and
# a plot title; everything else (grouping, styling, adaptive x-axis, faceted
# grid layout) is shared. "target"/"ylim" are optional and only make sense
# for metrics with a known throughput-dependent expectation.
METRICS = {
    "total_time": {
        "title": "Total Time",
        "extract": lambda b: b["data"]["analysis_end_ts"] - b["data"]["producer_begin_ts"],
        "ylabel": "average tot time (s)",
        "target": lambda throughput: 1920 / throughput,
        "target_label": "target time",
        "ylim": lambda throughput: (0, 3840 / throughput),
    },
    "achieved_throughput_MBps": {
        "title": "Achieved Throughput (MB/s)",
        "extract": _achieved_throughput_MBps,
        "ylabel": "achieved throughput (MB/s)",
        "target": lambda throughput: throughput,
        "target_label": "target throughput",
        "ylim": lambda throughput: (0, throughput * 1.3),
    },
    "avg_processing_time_ms": {
        "title": "Avg Processing Time (ms)",
        "extract": lambda b: np.mean(_record_values(b, "processing_time_ms")),
        "ylabel": "average processing time (ms)",
        "target": None,
        "ylim": None,
    },
    "avg_net_latency_ms": {
        "title": "Avg Network Latency (ms)",
        "extract": lambda b: np.mean(_record_values(b, "batch_latency_ms")),
        "ylabel": "average network latency (ms)",
        "target": None,
        "ylim": None,
    },
}

_SHORT_NAMES = {"throughput": "MBps", "n_scans": "SpB", "partition_worker_ratio": "pwr", "n_workers": "ws"}
_BAR_COLOR = "#3B78A6"
_ACCENT_COLOR = "#22303F"


def _draw_panel(ax, points, variable, metric_spec, panel_fixed_dict):
    """Draw one metric-vs-variable bar panel (bars + labels + y-scale) into a
    given Axes, plus a legend listing mean +/- SEM for each bar. Used as one
    subplot of a faceted grid - the shared "<metric> vs <variable>" title
    lives once on the figure (see plot_metric_vs), not repeated per panel,
    to avoid titles colliding between columns."""
    points = sorted(points, key=lambda p: p["x"])
    xs = [p["x"] for p in points]
    means = [p["mean"] for p in points]
    errors = [p["std"] for p in points]
    n_points = len(points)

    positions = np.arange(n_points)

    bars = ax.bar(
        positions, means, yerr=errors, capsize=4, width=0.6,
        color=_BAR_COLOR, edgecolor="white", linewidth=1.2,
        error_kw=dict(ecolor=_ACCENT_COLOR, elinewidth=1.2, capthick=1.2),
        zorder=3,
    )

    ax.set_xticks(positions)
    ax.set_xticklabels([str(x) for x in xs])
    ax.set_xlim(-0.7, n_points - 0.3)

    ax.set_xlabel(variable, fontsize=11)
    ax.set_ylabel(metric_spec["ylabel"], fontsize=11)
    # Only the per-panel fixed parameters here - the "<metric> vs <variable>"
    # part is a single figure-level suptitle (see plot_metric_vs).
    ax.set_title(
        " ".join(f"{k}={v}" for k, v in panel_fixed_dict.items()),
        fontsize=10.5, color="#666666", pad=10,
    )

    throughput = panel_fixed_dict.get("throughput")

    target_line = None
    if metric_spec["ylim"] is not None:
        if throughput is None:
            raise ValueError("throughput must be fixed to use this metric's y-scale")
        ax.set_ylim(*metric_spec["ylim"](throughput))
        if metric_spec["target"] is not None:
            target = metric_spec["target"](throughput)
            target_line = ax.axhline(target, linestyle="--", color=_ACCENT_COLOR, linewidth=1.3, zorder=4)
    else:
        y_max = max(m + e for m, e in zip(means, errors)) * 1.2
        ax.set_ylim(0, y_max if y_max > 0 else 1)

    for bar, p in zip(bars, points):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() / 2, str(len(p["files"])),
            ha="center", va="center", fontsize=10, fontweight="bold", color="white",
        )

    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.25, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Legend: mean +/- SEM (SEM = std / sqrt(n runs)) for each bar, plus the
    # target line if this metric has one.
    legend_handles, legend_labels = [], []
    if target_line is not None:
        legend_handles.append(target_line)
        legend_labels.append(metric_spec["target_label"])
    for bar, p in zip(bars, points):
        n = len(p["files"])
        sem = p["std"] / math.sqrt(n) if n > 0 else float("nan")
        legend_handles.append(bar)
        legend_labels.append(f"{variable}={p['x']}: {p['mean']:.2f} \u00b1 {sem:.2f}")
    ax.legend(legend_handles, legend_labels, loc="best", fontsize=7.5, frameon=False)


def plot_metric_vs(variable, metric="total_time", facet_variable="n_scans",
                    benchmark_dir="benchmarks", output_dir="results"):
    """Plot `metric` vs `variable`, one grid image per remaining fixed
    combination, faceting into subplots over `facet_variable` (e.g. one grid
    per throughput/partition_worker_ratio combo, with one subplot per
    n_scans value) - instead of a separate image per n_scans value."""
    assert variable in {"throughput", "n_scans", "partition_worker_ratio", "n_workers"}
    assert facet_variable in {"throughput", "n_scans", "partition_worker_ratio", "n_workers"}
    assert facet_variable != variable
    assert metric in METRICS, f"Unknown metric {metric!r}, choose from {sorted(METRICS)}"
    metric_spec = METRICS[metric]

    output_dir = Path(output_dir) / metric
    output_dir.mkdir(parents=True, exist_ok=True)

    benchmarks = load_benchmark_files(directory=benchmark_dir, verify_content=True)

    def partition_worker_ratio(b):
        n_parts, n_workers = b["n_partitions"], b["n_workers"]
        assert n_parts % n_workers == 0, (
            f"n_partitions ({n_parts}) is not a multiple of n_workers ({n_workers}) "
            f"for {b['path']}"
        )
        return n_parts // n_workers

    get_value = {
        "throughput": lambda b: b["throughput_MB"],
        "n_scans": lambda b: b["data"]["n_scans_per_batch"],
        "partition_worker_ratio": partition_worker_ratio,
        "n_workers": lambda b: b["n_workers"],
    }

    fixed_variables = [v for v in get_value if v not in (variable, facet_variable)]

    # panels[outer_fixed][facet_value] = list of {"x", "mean", "std", "files"}
    grouped = defaultdict(list)
    for b in benchmarks:
        x = get_value[variable](b)
        facet = get_value[facet_variable](b)
        outer_fixed = tuple(get_value[v](b) for v in fixed_variables)
        value = metric_spec["extract"](b)
        grouped[(outer_fixed, facet, x)].append({"value": value, "file": b["path"]})

    panels = defaultdict(lambda: defaultdict(list))
    for (outer_fixed, facet, x), runs in grouped.items():
        values = [r["value"] for r in runs]
        panels[outer_fixed][facet].append({
            "x": x,
            "mean": np.mean(values),
            "std": np.std(values),
            "files": [r["file"] for r in runs],
        })

    for outer_fixed, facets_dict in panels.items():
        outer_fixed_dict = dict(zip(fixed_variables, outer_fixed))
        facet_values = sorted(facets_dict)
        n_facets = len(facet_values)

        ncols = 2 if n_facets > 1 else 1
        nrows = math.ceil(n_facets / ncols)

        fig_w = max(4.5, 1.6 * max(len(v) for v in facets_dict.values()) + 2.2)
        fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w * ncols, 5.6 * nrows), squeeze=False)
        axes_flat = axes.flatten()

        for ax, facet_value in zip(axes_flat, facet_values):
            panel_fixed_dict = dict(outer_fixed_dict)
            panel_fixed_dict[facet_variable] = facet_value
            # Keep display order: throughput, n_scans, partition_worker_ratio, n_workers
            panel_fixed_dict = {k: panel_fixed_dict[k] for k in get_value if k in panel_fixed_dict}
            _draw_panel(ax, facets_dict[facet_value], variable, metric_spec, panel_fixed_dict)

        # Hide any unused subplot slots (e.g. 3 facets in a 2x2 grid).
        for ax in axes_flat[n_facets:]:
            ax.set_visible(False)

        # Single shared title for the whole grid (was previously repeated on
        # every subplot, which collided with the neighboring subplot's title
        # in a multi-column layout).
        fig.suptitle(f"{metric_spec['title']} vs {variable}", fontsize=16, fontweight="bold")

        fig.tight_layout()
        fig.subplots_adjust(top=1 - 0.77 / (5.6 * nrows), wspace=0.3, hspace=0.5)

        short_names = _SHORT_NAMES
        fixed_string = "__".join(f"{k}{v}{short_names[k]}" for k, v in outer_fixed_dict.items())
        filename = f"{metric}_vs_{variable}__{fixed_string}.png"

        fig.savefig(output_dir / filename, dpi=300)
        plt.close(fig)

        txt_filename = filename.replace(".png", ".txt")
        with (output_dir / txt_filename).open("w") as f:
            f.write(f"metric={metric}\n")
            f.write(f"variable={variable}\n")
            f.write(f"facet_variable={facet_variable}\n")
            for k, v in outer_fixed_dict.items():
                f.write(f"{k}={v}\n")
            f.write("\n")

            for facet_value in facet_values:
                f.write(f"### {facet_variable}={facet_value}\n")
                for p in sorted(facets_dict[facet_value], key=lambda p: p["x"]):
                    n = len(p["files"])
                    sem = p["std"] / math.sqrt(n) if n > 0 else float("nan")
                    f.write(f"{variable}={p['x']}\n")
                    f.write(f"mean_{metric}={p['mean']}\n")
                    f.write(f"std_{metric}={p['std']}\n")
                    f.write(f"sem_{metric}={sem}\n")
                    for file in p["files"]:
                        f.write(f"    {file}\n")
                    f.write("\n")

In [88]:
for metric in ["total_time", "achieved_throughput_MBps", "avg_processing_time_ms", "avg_net_latency_ms"]:
    plot_metric_vs("n_workers", metric)
    plot_metric_vs("partition_worker_ratio", metric)

# Latency distributions (network/Kafka vs Dask FFT computation)

In [89]:
import matplotlib.pyplot as plt
import numpy as np


def _draw_latency_hist(ax, values, title, color, bins):
    values = np.asarray(values)
    mean = values.mean()
    p50 = np.percentile(values, 50)
    p95 = np.percentile(values, 95)

    ax.hist(values, bins=bins, color=color, edgecolor="black", alpha=0.75)

    mean_line = ax.axvline(mean, color="red", linestyle="--", linewidth=1.3)
    p50_line = ax.axvline(p50, color="orange", linestyle="--", linewidth=1.3)
    p95_line = ax.axvline(p95, color="green", linestyle="--", linewidth=1.3)

    ax.legend(
        [mean_line, p50_line, p95_line],
        [f"Mean: {mean:.2f} ms", f"P50: {p50:.2f} ms", f"P95: {p95:.2f} ms"],
        loc="upper right", fontsize=9,
    )

    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Latency (ms)")
    ax.set_ylabel("Frequency (Number of Packets)")
    ax.grid(True, alpha=0.3)

    return {"mean": float(mean), "p50": float(p50), "p95": float(p95), "n": len(values)}


def plot_latency_histograms(
    benchmark_dir="benchmarks", output_dir="results/latencies",
    throughput=None, n_scans_per_batch=None, n_workers=None, n_partitions=None,
    bins=30,
):
    """One latency-histogram figure PER benchmark run (not pooled across
    runs): network+Kafka delivery latency (batch_latency_ms) side by side
    with the Dask worker's own FFT computation time (processing_time_ms),
    both from that single run's records. Filters work the same as
    load_benchmark_files (None = any) and just narrow down which runs get a
    figure."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    benchmarks = load_benchmark_files(
        directory=benchmark_dir, throughput=throughput, n_scans_per_batch=n_scans_per_batch,
        n_workers=n_workers, n_partitions=n_partitions, verify_content=True,
    )

    for b in benchmarks:
        net_latencies = _record_values(b, "batch_latency_ms")
        processing_times = _record_values(b, "processing_time_ms")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

        net_stats = _draw_latency_hist(axes[0], net_latencies, "Network & Kafka Pipeline Latency", "#6D8FC7", bins)
        proc_stats = _draw_latency_hist(axes[1], processing_times, "Dask Worker FFT Computation Latency", "#7DB893", bins)

        run_tag = (
            f"{b['throughput_MB']}MBps__{b['n_scans_per_batch']}SpB__"
            f"{b['n_workers']}ws__{b['n_partitions']}parts__{b['path'].stem}"
        )
        fig.suptitle(run_tag, fontsize=10, color="#666666", y=1.02)
        fig.tight_layout()

        filename = f"latency_histograms__{run_tag}.png"
        fig.savefig(output_dir / filename, dpi=300, bbox_inches="tight")
        plt.close(fig)

        with (output_dir / filename.replace(".png", ".txt")).open("w") as f:
            f.write(f"run={b['path']}\n")
            f.write(f"throughput={b['throughput_MB']}\n")
            f.write(f"n_scans={b['n_scans_per_batch']}\n")
            f.write(f"n_workers={b['n_workers']}\n")
            f.write(f"n_partitions={b['n_partitions']}\n")
            f.write("\n")
            f.write("batch_latency_ms: " + ", ".join(f"{k}={v}" for k, v in net_stats.items()) + "\n")
            f.write("processing_time_ms: " + ", ".join(f"{k}={v}" for k, v in proc_stats.items()) + "\n")

In [90]:
plot_latency_histograms()